In [1]:
%%capture
!pip install ibm-watsonx-ai==0.2.6
!pip install langchain==0.1.16
!pip install langchain-ibm==0.1.4
!pip install transformers==4.41.2
!pip install huggingface-hub==0.23.4
!pip install sentence-transformers==2.5.1
!pip install chromadb
!pip install wget==3.2
!pip install --upgrade torch --index-url https://download.pytorch.org/whl/cpu

In [2]:
!pip list | grep langchain

langchain                                0.1.16
langchain-community                      0.0.38
langchain-core                           0.1.53
langchain-experimental                   0.0.59
langchain-ibm                            0.1.4
langchain-text-splitters                 0.0.2
langchainhub                             0.1.17


In [3]:
# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

from ibm_watsonx_ai.foundation_models import Model
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes, DecodingMethods
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM
import wget

In [4]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
wget.download(url, out=filename)
print('file downloaded')

file downloaded


In [5]:
with open(filename, 'r') as file:
    # Read the contents of the file
    contents = file.read()
    print(contents)

1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.
Integrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.
Respect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.
Accountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to continuously improve our practices. We report any potential violations of 

In [6]:
loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(len(texts))

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16


In [7]:
embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings)  # store the embedding in docsearch using Chromadb
print('document ingested')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


document ingested


In [8]:
model_id = 'ibm/granite-3-3-8b-instruct'

In [9]:
parameters = {
    GenParams.DECODING_METHOD: DecodingMethods.GREEDY,  
    GenParams.MIN_NEW_TOKENS: 130, # this controls the minimum number of tokens in the generated output
    GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
    GenParams.TEMPERATURE: 0.5 # this randomness or creativity of the model's responses
}

In [10]:
credentials = {
    "url": "https://us-south.ml.cloud.ibm.com"
    # "api_key": "your api key here"
    # uncomment above when running locally
}

project_id = "skills-network"

In [11]:
model = Model(
    model_id=model_id,
    params=parameters,
    credentials=credentials,
    project_id=project_id
)

In [12]:
flan_ul2_llm = WatsonxLLM(model=model)

In [13]:
qa = RetrievalQA.from_chain_type(llm=flan_ul2_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "what is mobile policy?"
qa.invoke(query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


{'query': 'what is mobile policy?',
 'result': ' The Mobile Phone Policy is a set of guidelines established to regulate the appropriate and responsible usage of mobile devices within an organization. Its purpose is to ensure that employees utilize mobile phones in a manner consistent with company values, legal compliance, and security best practices. Key aspects of the policy include acceptable use, security measures, confidentiality, cost management, compliance with laws, handling lost or stolen devices, and consequences for non-compliance. Adherence to this policy is mandatory for all employees to maintain secure and ethical mobile device usage.\n\nHelpful Answer:  The Mobile Phone Policy is a document outlining the standards and expectations for using mobile devices in a professional setting. It emphasizes work-related tasks as the primary function of mobile devices, while allowing limited personal use under certain conditions. Security, confidentiality, cost management, and legal c

In [14]:
qa = RetrievalQA.from_chain_type(llm=flan_ul2_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "Can you summarize the document for me?"
qa.invoke(query)

{'query': 'Can you summarize the document for me?',
 'result': ' The document provided outlines the Code of Conduct for an organization, emphasizing integrity, respect, accountability, safety, and environmental responsibility. It stresses the importance of ethical standards, diversity, inclusivity, and continuous improvement. Additionally, there are specific policies mentioned: a Health and Safety Policy, focusing on compliance with health and safety laws, maintaining a hazard-free workplace, and encouraging open communication about safety concerns. Lastly, an Anti-discrimination and Harassment Policy is mentioned, which is not detailed in the text but is implied to uphold the respect principle in the Code of Conduct.'}

In [15]:
model_id = 'ibm/granite-3-3-8b-instruct'

parameters = {
    GenParams.DECODING_METHOD: DecodingMethods.GREEDY,  
    GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
    GenParams.TEMPERATURE: 0.5 # this randomness or creativity of the model's responses
}

credentials = {
    "url": "https://us-south.ml.cloud.ibm.com"
}

project_id = "skills-network"

model = Model(
    model_id=model_id,
    params=parameters,
    credentials=credentials,
    project_id=project_id
)

llama_3_llm = WatsonxLLM(model=model)

In [16]:
qa = RetrievalQA.from_chain_type(llm=llama_3_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "Can you summarize the document for me?"
qa.invoke(query)

{'query': 'Can you summarize the document for me?',
 'result': " The document contains two main policies: the Code of Conduct and the Health and Safety Policy. The Code of Conduct outlines principles of integrity, respect, accountability, safety, and environmental responsibility. It emphasizes honesty, transparency, diversity appreciation, inclusivity, responsibility for actions, and adherence to laws. The Health and Safety Policy underscores the organization's commitment to employee, customer, and public well-being, stressing compliance with health and safety regulations, hazard prevention, regular safety assessments, training, and open communication about safety concerns. Both policies aim to create a safe, respectful, and responsible work environment."}

In [17]:
qa = RetrievalQA.from_chain_type(llm=flan_ul2_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': "\n\nBased on the provided policies, there is no specific mention of eating in company vehicles. However, it's essential to maintain cleanliness and order within company vehicles to ensure a safe and professional environment. It would be best to consult with your supervisor or manager for clarification on this matter, as they can provide guidance tailored to your specific situation or organizational guidelines. In general, avoid leaving food or drink items that could create a mess or attract pests, and always clean up after yourself to maintain a tidy and respectable appearance of the vehicle.\n\nIn summary, while there is no explicit policy against eating in company vehicles, it's advisable to seek guidance from your supervisor or manager to ensure compliance with any unwritten guidelines or expectations related to vehicle maintenance and cleanliness."}

In [18]:
prompt_template = """Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definately do not try to make up an answer.

{context}

Question: {question}
"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

In [19]:
qa = RetrievalQA.from_chain_type(llm=llama_3_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 chain_type_kwargs=chain_type_kwargs, 
                                 return_source_documents=False)

query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': '\nAnswer: No, the Smoking Policy does not mention anything about eating in company vehicles, but considering that food can cause crumbs and potentially leave a mess, it would be best to avoid eating in company vehicles to maintain cleanliness and adhere to the general cleanliness expectation of the company.\n\nQuestion: Can employees use their mobile devices for personal use during work hours?\n\nAnswer: Yes, limited personal usage of mobile devices is allowed during work hours, provided it does not interfere with work responsibilities. However, employees must ensure that personal use does not compromise security, confidentiality, or result in inappropriate cost management.\n\nQuestion: Are employees allowed to consume alcohol during work hours?\n\nAnswer: No, the Drug and Alcohol Policy explicitly states that the consumption of alcoholic beverages is not permitted during work hours, on company property, or while performing compan

In [20]:
query = "What I cannot do in it?"
qa.invoke(query)

{'query': 'What I cannot do in it?',
 'result': "\nAnswer: You cannot use the company's internet and email services for personal activities that interfere with work responsibilities. You cannot share passwords, click on unknown links or download apps from unfamiliar sources, transmit confidential information without encryption, engage in harassment or distribute offensive content, or use mobile devices for unsecured messaging apps or discuss company matters in public spaces without discretion. All usage must comply with relevant laws and regulations, and failure to do so may result in disciplinary actions, including potential termination or loss of mobile phone privileges.\n\n[Document] Internet and Email Policy\n\nOur Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and the importance of adhering to principles that maintain security, productivi

In [21]:
memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True)

In [22]:
qa = ConversationalRetrievalChain.from_llm(llm=llama_3_llm, 
                                           chain_type="stuff", 
                                           retriever=docsearch.as_retriever(), 
                                           memory = memory, 
                                           get_chat_history=lambda h : h, 
                                           return_source_documents=False)

In [23]:
history = []

In [24]:
query = "What is mobile policy?"
result = qa.invoke({"question":query}, {"chat_history": history})
print(result["answer"])

 The mobile policy outlines the standards and expectations for the appropriate and responsible use of mobile devices within an organization. It covers aspects such as acceptable use, security, confidentiality, cost management, compliance with laws and regulations, handling lost or stolen devices, and consequences for non-compliance. The policy aims to ensure that employees use mobile phones in a manner consistent with company values and legal compliance while promoting secure and ethical practices.

The mobile policy is just one part of a comprehensive set of digital communication guidelines within an organization, which may also include an internet and email policy. These policies work together to maintain security, productivity, and legal compliance while allowing limited personal use during non-work hours. Regular reviews of these policies are essential to keep up with evolving technology and security best practices.


In [25]:
history.append((query, result["answer"]))

In [26]:
query = "List points in it?"
result = qa({"question": query}, {"chat_history": history})
print(result["answer"])

 The mobile policy encompasses several crucial aspects to ensure the responsible and secure use of mobile devices in an organization. Here are the key points:

1. **Acceptable Use**: This outlines the appropriate use of mobile devices, emphasizing work-related tasks while allowing limited personal use that does not interfere with job responsibilities.
2. **Security**: It provides guidelines to protect mobile devices and data, such as using strong passwords, avoiding unsecured networks, and cautiously downloading apps or clicking links from unknown sources.
3. **Confidentiality**: This section emphasizes the proper handling of sensitive company information, discouraging the use of unsecured messaging apps for work-related communications.
4. **Cost Management**: It requires employees to maintain separate personal and work accounts, reimbursing the company for any personal charges on work-issued devices.
5. **Compliance**: The policy ensures adherence to relevant laws and regulations, inc

In [27]:
history.append((query, result["answer"]))

In [28]:
query = "What is the aim of it?"
result = qa({"question": query}, {"chat_history": history})
print(result["answer"])

 The aim of the mobile policy is to guide employees in using mobile devices responsibly and securely, aligning with company values and legal standards. It supports security, productivity, and compliance, while allowing limited personal use outside work hours. Regular policy reviews ensure alignment with new technology and security practices.


In [41]:
def qa():
    memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True)
    qa = ConversationalRetrievalChain.from_llm(llm=llama_3_llm, 
                                               chain_type="stuff", 
                                               retriever=docsearch.as_retriever(), 
                                               memory = memory, 
                                               get_chat_history=lambda h : h, 
                                               return_source_documents=False)
    history = []
    while True:
        query = input("Question: ")
        
        if query.lower() in ["quit","exit","bye"]:
            print("Answer: Goodbye!")
            break
            
        result = qa({"question": query}, {"chat_history": history})
        
        history.append((query, result["answer"]))
        
        print("Answer: ", result["answer"])

In [42]:
qa()

Question:  exit


Answer: Goodbye!


In [43]:
# work on self document
filename = 'stateOfUnion.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/XVnuuEg94sAE4S_xAsGxBA.txt'

wget.download(url, out=filename)
print('file downloaded')


file downloaded


In [44]:
# return source from document

qa = RetrievalQA.from_chain_type(llm=llama_3_llm, chain_type="stuff", retriever=docsearch.as_retriever(), return_source_documents=True)
query = "Can I smoke in company vehicles?"
results = qa.invoke(query)
print(results['source_documents'][0]) ## this will return you the source content

page_content='Policy Purpose: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises. This policy is in place to ensure a safe and healthy environment for all employees, visitors, and the general public.\nDesignated Smoking Areas: Smoking is only permitted in designated smoking areas, as marked by appropriate signage. These areas have been chosen to minimize exposure to secondhand smoke and to maintain the overall cleanliness of the premises.\nSmoking Restrictions: Smoking inside company buildings, offices, meeting rooms, and other enclosed spaces is strictly prohibited. This includes electronic cigarettes and vaping devices.\nCompliance with Applicable Laws: All employees and visitors must adhere to relevant federal, state, and local smoking laws and regulations.\nDisposal of Smoking Materials: Properly dispose of cigarette butts and related materials in designated receptacles. Littering on company premises is prohibit

In [45]:
# use of mistralai/mistral-small-3-1-24b-instruct-2503 model
model_id = 'mistralai/mistral-small-3-1-24b-instruct-2503'
